# Giai đoạn 6 — Đánh giá kết quả train (bản cải tiến).

Giai đoạn 6 — Đánh giá kết quả train (bản cải tiến).
Input : datas/05_models/best_*.pkl + datas/04_split/{X_test_*, y_test_*, segments_test.csv}
Output: datas/06_evaluation/{metrics.csv, segment_metrics.csv, predictions.csv} + figures/
- Hiệu chỉnh count: clip >= 0 + làm tròn số nguyên
- Metric R2/RMSE/RMSLE/MAPE (mask y_true != 0, không dùng biến global)
- So sánh (casual+registered) vs baseline cnt trực tiếp
- Cải tiến: metrics theo phân khúc (giờ cao điểm, cuối tuần, thời tiết xấu, đêm)
  để phát hiện model tốt toàn cục nhưng tệ lúc cần điều xe nhất
Chạy: python src/06_evaluate.py

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_percentage_error
# Cấu hình đường dẫn (inline để notebook chạy độc lập, không cần config.py)
ROOT = Path.cwd()
if not (ROOT / "datas").exists() and (ROOT.parent / "datas").exists():
    ROOT = ROOT.parent  # khi kernel chạy từ trong thư mục src/
RAW_CSV = ROOT / "datas" / "hour.csv"
EDA_DIR = ROOT / "datas" / "01_eda"
CLEANED_DIR = ROOT / "datas" / "02_cleaned"
FEATURES_DIR = ROOT / "datas" / "03_features"
SPLIT_DIR = ROOT / "datas" / "04_split"
MODELS_DIR = ROOT / "datas" / "05_models"
EVAL_DIR = ROOT / "datas" / "06_evaluation"
CAT_COLS = ["season", "yr", "mnth", "hr", "holiday", "weekday", "workingday", "weathersit", "time_period"]
TEST_SIZE = 0.2
RANDOM_STATE = 42

FIG = EVAL_DIR / "figures"


In [2]:
def postprocess(y_pred):
    """Count data: không âm + số nguyên."""
    return np.clip(np.rint(np.asarray(y_pred).flatten()), 0, None)


In [3]:
def test_metrics(y_true, y_pred):
    y_true = np.asarray(y_true).flatten()
    y_pred = postprocess(y_pred)
    mask = y_true != 0
    mape = (mean_absolute_percentage_error(y_true[mask], y_pred[mask]) * 100
            if mask.sum() else float("nan"))
    return {"R2": round(float(r2_score(y_true, y_pred)), 4),
            "RMSE": round(float(np.sqrt(mean_squared_error(y_true, y_pred))), 2),
            "RMSLE": round(float(np.sqrt(mean_squared_error(
                np.log1p(y_true), np.log1p(y_pred)))), 4),
            "MAPE": round(float(mape), 2),
            "MAE": round(float(np.mean(np.abs(y_true - y_pred))), 2)}


In [4]:
def plot_sorted(y_true, y_pred, name):
    y_true = np.asarray(y_true).flatten()
    y_pred = postprocess(y_pred)
    idx = np.argsort(y_true)
    plt.figure(figsize=(12, 4))
    plt.plot(y_true[idx], label="true", linewidth=1)
    plt.plot(y_pred[idx], label="pred", linewidth=1)
    plt.title(f"Sorted pred vs true — {name}")
    plt.legend()
    plt.tight_layout()
    plt.savefig(FIG / f"sorted_{name}.png", dpi=120)
    plt.close()


In [5]:
def plot_residual(y_true, y_pred, name):
    y_true = np.asarray(y_true).flatten()
    y_pred = postprocess(y_pred)
    res = y_true - y_pred
    _, ax = plt.subplots(1, 2, figsize=(12, 4))
    sns.scatterplot(x=y_pred, y=res, alpha=0.3, ax=ax[0])
    ax[0].axhline(0, color="red")
    ax[0].set_title(f"Residual vs pred — {name}")
    sns.histplot(res, kde=True, ax=ax[1])
    ax[1].set_title(f"Residual dist — {name}")
    plt.tight_layout()
    plt.savefig(FIG / f"residual_{name}.png", dpi=120)
    plt.close()


In [6]:
def segment_masks(seg):
    hr = seg["hr"].astype(int)
    wd = seg["workingday"].astype(int) if "workingday" in seg else None
    return {
        "peak": hr.isin([7, 8, 9, 17, 18, 19]),
        "weekend": (seg["is_weekend"].astype(int) == 1) if "is_weekend" in seg else hr.isin([]),
        "bad_weather": (seg["weathersit"].astype(int) >= 3) if "weathersit" in seg else hr.isin([]),
        "night": hr.isin([0, 1, 2, 3, 4, 5]),
        "workingday": (wd == 1) if wd is not None else hr.isin([]),
    }


In [7]:
def main():
    EVAL_DIR.mkdir(parents=True, exist_ok=True)
    FIG.mkdir(parents=True, exist_ok=True)
    best_c = joblib.load(MODELS_DIR / "best_casual.pkl")
    best_r = joblib.load(MODELS_DIR / "best_registered.pkl")
    best_cnt = joblib.load(MODELS_DIR / "best_cnt.pkl")

    out, preds = [], {}
    for tag, model in [("casual", best_c), ("registered", best_r), ("cnt", best_cnt)]:
        Xt = pd.read_csv(SPLIT_DIR / f"X_test_{tag}.csv")
        yt = pd.read_csv(SPLIT_DIR / f"y_test_{tag}.csv").values.ravel()
        yp = postprocess(model.predict(Xt))
        out.append({"target": tag, **test_metrics(yt, yp)})
        preds[tag] = (yt, yp)
        plot_sorted(yt, yp, tag)
        plot_residual(yt, yp, tag)

    # Cộng 2 mô hình so với baseline cnt trực tiếp (chung index thời gian)
    yc = preds["casual"][0] + preds["registered"][0]
    yp_sum = postprocess(preds["casual"][1] + preds["registered"][1])
    out.append({"target": "sum_casual_registered", **test_metrics(yc, yp_sum)})
    plot_sorted(yc, yp_sum, "sum_casual_registered")
    plot_residual(yc, yp_sum, "sum_casual_registered")

    pd.DataFrame(out).to_csv(EVAL_DIR / "metrics.csv", index=False)

    # Metrics theo phân khúc cho mô hình tổng (quyết định điều xe)
    seg = pd.read_csv(SPLIT_DIR / "segments_test.csv")
    seg_rows = [{"segment": "all", "n": len(yc), **test_metrics(yc, yp_sum)}]
    for name, mask in segment_masks(seg).items():
        m = np.asarray(mask)
        if m.sum() == 0:
            continue
        seg_rows.append({"segment": name, "n": int(m.sum()),
                         **test_metrics(yc[m], yp_sum[m])})
    pd.DataFrame(seg_rows).to_csv(EVAL_DIR / "segment_metrics.csv", index=False)

    pd.DataFrame({"y_true_sum": yc, "y_pred_sum": yp_sum,
                  "y_true_cnt": preds["cnt"][0],
                  "y_pred_cnt": preds["cnt"][1]}).to_csv(
        EVAL_DIR / "predictions.csv", index=False)
    print("=== Metrics tổng ===")
    print(pd.DataFrame(out).to_string(index=False))
    print("\n=== Metrics theo phân khúc (sum model) ===")
    print(pd.DataFrame(seg_rows).to_string(index=False))
    print(f"OK -> {EVAL_DIR}")


In [8]:
if __name__ == "__main__":
    main()


=== Metrics tổng ===
               target     R2  RMSE  RMSLE  MAPE   MAE
               casual 0.9424 13.44 0.4663 43.39  8.17
           registered 0.9507 41.78 0.2940 24.96 25.98
                  cnt 0.9487 49.95 0.5771 56.22 33.70
sum_casual_registered 0.9562 46.15 0.2853 24.05 28.94

=== Metrics theo phân khúc (sum model) ===
    segment    n     R2  RMSE  RMSLE  MAPE   MAE
        all 3476 0.9562 46.15 0.2853 24.05 28.94
       peak  870 0.9225 69.19 0.2303 16.92 48.18
    weekend 1008 0.9563 43.88 0.2795 22.91 29.06
bad_weather  238 0.8586 67.94 0.4507 45.51 43.56
      night  861 0.8389 14.96 0.4311 48.52  8.42
 workingday 2349 0.9562 47.26 0.2827 23.86 28.93
OK -> /mnt/d/Documents/UIT/HK2/CKIE313/datas/06_evaluation


## So sánh MAE các mô hình (từ `cv_results_*.csv`)

In [9]:
def plot_cv_bars(tag):
    cv = pd.read_csv(MODELS_DIR / f"cv_results_{tag}.csv").sort_values("mae_mean")
    plt.figure(figsize=(8, 4))
    ax = sns.barplot(data=cv, x="mae_mean", y="model", hue="model", palette="Blues_r")
    plt.title(f"So sánh MAE (CV mean) — {tag} (thấp hơn tốt hơn)")
    plt.xlabel("Mean Absolute Error (MAE)")
    for p in ax.patches:
        w = p.get_width()
        ax.annotate(f"{w:.2f}", (w, p.get_y() + p.get_height() / 2),
                    ha="left", va="center", xytext=(5, 0), textcoords="offset points")
    plt.tight_layout()
    plt.savefig(FIG / f"cv_bars_{tag}.png", dpi=120)
    plt.show()
    plt.close()

for _tag in ["casual", "registered", "cnt"]:
    plot_cv_bars(_tag)


/tmp/ipykernel_6833/415169672.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


/tmp/ipykernel_6833/415169672.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


/tmp/ipykernel_6833/415169672.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
